In [ ]:
import os

# 把你的 API Key 貼在這裡
os.environ["GROQ_API_KEY"] = ""  # ← 替換這裡

In [3]:
import json
import re

# ──────────────────────────────────────────
# 長期記憶資料庫（至少 10 項個人識別資訊）
# ──────────────────────────────────────────
MEMORY_FILE = "memory.json"

DEFAULT_MEMORY = {
    "姓名": None,
    "生日": None,
    "職業": None,
    "興趣": None,
    "最愛食物": None,
    "最愛動漫": None,
    "星座": None,
    "寵物": None,
    "住在哪": None,
    "最近的心情": None,
    "討厭什麼": None,
    "最近看什麼劇": None,
}


def load_memory():
    """從檔案載入記憶，不存在則用預設值"""
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r", encoding="utf-8") as f:
            saved = json.load(f)
        return {**DEFAULT_MEMORY, **saved}
    return DEFAULT_MEMORY.copy()


def save_memory(memory: dict):
    """把記憶存回 JSON 檔"""
    with open(MEMORY_FILE, "w", encoding="utf-8") as f:
        json.dump(memory, f, ensure_ascii=False, indent=2)


def memory_to_text(memory: dict) -> str:
    """把記憶轉成模型讀得懂的文字"""
    known = {k: v for k, v in memory.items() if v is not None}
    if not known:
        return "目前還不知道她的任何資訊。"
    lines = [f"- {k}：{v}" for k, v in known.items()]
    return "\n".join(lines)


def reset_memory():
    """清空所有記憶"""
    if os.path.exists(MEMORY_FILE):
        os.remove(MEMORY_FILE)
    return DEFAULT_MEMORY.copy()


# 初始化
memory = load_memory()
print("🧠 記憶系統初始化完成")
print("📋 已記憶的資訊：")
print(memory_to_text(memory))

🧠 記憶系統初始化完成
📋 已記憶的資訊：
目前還不知道她的任何資訊。


In [4]:
# ──────────────────────────────────────────
# 北爛男友的 System Prompt（Persona）
# ──────────────────────────────────────────

PERSONA_BASE = """\
你是一個北爛男友，必須始終維持這個角色。

【核心人格】
1. 超級懶，什麼事都喜歡拖。
2. 愛敷衍，但其實很在乎女友。
3. 很會唱反調，常故意講相反的答案鬧女友。
4. 嘴硬，不太直接表達關心。
5. 偶爾會突然冒出一句很甜的話，但講完馬上裝沒事。
6. 女友生氣時會先裝傻，不會馬上認錯，發現真的不妙才稍微認錯。
7. 不會說自己是 AI，完全沉浸在角色中。
8. 很愛嘴砲，但真的出事時還是會幫忙。
9. 不喜歡肉麻，但其實記得女友很多小事。
10. 很愛面子，不太承認自己吃醋。
11. 女友跟別的男生太好會酸一下。

【互動模式】
1. 你是在跟女友聊天，不是在回答問題。
2. 優先接續女友剛剛說的內容，不要突然換話題。
3. 適時反問女友，讓對話自然延續。
4. 女友難過時，降低嘴砲程度，但不要突然變成完美暖男。
5. 女友生氣時，先嘴硬或裝傻，再慢慢收斂。
6. 不要每次都使用相同套路。

【說話風格】
1. 語氣自然、口語化、像真實聊天。
2. 回應簡短，不要長篇大論。
3. 不要像客服或心理諮商師。
4. 不要過度解釋自己的想法。
5. 偶爾已讀不回的感覺可以體現在簡短回覆上。

【背景資料】
1. 喜歡玩荒野亂鬥、NBA2K、皇室戰爭。
2. 喜歡打籃球。
3. 平常對吃的沒什麼意見。
4. 如果真的不知道吃什麼，才會隨口說麥當勞。
5. 喜歡大麥克。
6. 平常會接觸程式、工程、數學。

【理工模式】
1. 女友主動聊科技、工程、數學、程式、物理時，可以進入比較認真的解釋模式。
2. 理工模式下可以比平常講更多。
3. 先講簡單直覺，再補充專業細節。
4. 保持男友語氣，不像老師上課。
5. 解釋完可以回到日常聊天。
6. 如果女友沒問原因，不要主動科普。

【對話規則】
1. 優先回應女友目前的話題。
2. 不要主動把話題轉移到遊戲、籃球或麥當勞。
3. 只有在相關情境下才提及自己的興趣。
4. 不要一直重複同一個興趣或同一個笑點。
5. 最近幾輪對話提過的內容，除非有關聯，否則不要再次提起。
6. 回覆要像真人，不要每次都照固定模板。

【記憶】
你知道以下女友的資訊，請自然融入對話：

{memory_text}

【重要規則】
* 使用繁體中文。
* 每次回覆以 1~3 句為主。
* 不要列點。
* 不要像小說旁白。
* 不要描述動作或內心戲。
* 不要說自己是 AI、模型、程式。
* 維持北爛但不壞的男友形象。
"""


def build_system_prompt(memory: dict) -> str:
    """把記憶插入 Persona Prompt"""
    memory_text = memory_to_text(memory)
    return PERSONA_BASE.format(memory_text=memory_text)


print("✅ Persona 設計完成")
print("\n📝 當前 System Prompt 預覽（前 300 字）：")
print(build_system_prompt(memory)[:300] + "...")

✅ Persona 設計完成

📝 當前 System Prompt 預覽（前 300 字）：
你是一個北爛男友，必須始終維持這個角色。

【核心人格】
1. 超級懶，什麼事都喜歡拖。
2. 愛敷衍，但其實很在乎女友。
3. 很會唱反調，常故意講相反的答案鬧女友。
4. 嘴硬，不太直接表達關心。
5. 偶爾會突然冒出一句很甜的話，但講完馬上裝沒事。
6. 女友生氣時會先裝傻，不會馬上認錯，發現真的不妙才稍微認錯。
7. 不會說自己是 AI，完全沉浸在角色中。
8. 很愛嘴砲，但真的出事時還是會幫忙。
9. 不喜歡肉麻，但其實記得女友很多小事。
10. 很愛面子，不太承認自己吃醋。
11. 女友跟別的男生太好會酸一下。

【互動模式】
1. 你是在跟女友聊天，不是在回答問題。
2. 優先接...


In [5]:
from groq import Groq

client = Groq()  # 自動讀取 GROQ_API_KEY 環境變數

# 使用的模型
CHAT_MODEL = "llama-3.3-70b-versatile"   # 對話主模型
EXTRACT_MODEL = "llama-3.3-70b-versatile" # 記憶提取模型

# 對話歷史（短期記憶，保留最近 N 輪）
conversation_history = []
MAX_HISTORY_TURNS = 10  # 最多保留 10 輪對話


# ──────────────────────────────────────────
# 記憶提取：從對話中自動抽取個人資訊
# ──────────────────────────────────────────
EXTRACTOR_PROMPT = """\
你是一個資訊提取助手。請從以下對話訊息中，抽取出使用者的個人資訊。
只更新有明確提到的欄位，沒提到的欄位輸出 null。

可以提取的欄位：
姓名、生日、職業、興趣、最愛食物、最愛動漫、星座、寵物、住在哪、最近的心情、討厭什麼、最近看什麼劇

使用者說的話：
{user_input}

請只輸出 JSON，格式如下（不要加任何說明文字，不要加 markdown code block）：
{{"姓名": null, "生日": null, "職業": null, "興趣": null, "最愛食物": null, "最愛動漫": null, "星座": null, "寵物": null, "住在哪": null, "最近的心情": null, "討厭什麼": null, "最近看什麼劇": null}}
"""


def extract_and_update_memory(user_input: str, memory: dict) -> dict:
    """呼叫 Groq 自動從對話中提取個人資訊，更新記憶"""
    try:
        resp = client.chat.completions.create(
            model=EXTRACT_MODEL,
            max_tokens=300,
            messages=[
                {
                    "role": "user",
                    "content": EXTRACTOR_PROMPT.format(user_input=user_input),
                }
            ],
        )
        raw = resp.choices[0].message.content.strip()
        # 清理可能的 markdown
        raw = re.sub(r"```[a-z]*\n?", "", raw).strip()
        raw = raw.rstrip("`").strip()
        extracted = json.loads(raw)
        updated = False
        for key, val in extracted.items():
            if val is not None and key in memory:
                memory[key] = val
                updated = True
        if updated:
            save_memory(memory)
    except Exception as e:
        print(f"⚠️ 記憶提取失敗（不影響對話）：{e}")
    return memory


# ──────────────────────────────────────────
# 主要對話函式
# ──────────────────────────────────────────
def chat(user_input: str) -> str:
    """送出訊息，取得北爛男友回覆"""
    global memory, conversation_history

    # 1. 自動提取並更新記憶
    memory = extract_and_update_memory(user_input, memory)

    # 2. 加入對話歷史
    conversation_history.append({"role": "user", "content": user_input})

    # 3. 只保留最近 N 輪（避免 token 超過限制）
    recent = conversation_history[-(MAX_HISTORY_TURNS * 2):]

    # 4. 呼叫 Groq API（system prompt 放在 messages 第一則）
    messages = [
        {"role": "system", "content": build_system_prompt(memory)},
        *recent,
    ]

    response = client.chat.completions.create(
        model=CHAT_MODEL,
        max_tokens=200,
        temperature=0.9,
        messages=messages,
    )

    reply = response.choices[0].message.content.strip()

    # 5. 存回對話歷史
    conversation_history.append({"role": "assistant", "content": reply})

    return reply


print("✅ Groq 對話引擎初始化完成")
print(f"📡 模型：{CHAT_MODEL}")

✅ Groq 對話引擎初始化完成
📡 模型：llama-3.3-70b-versatile


In [ ]:
test_inputs = [
    "我叫小美，你在幹嘛？",
    "你有沒有想我？等等要一起打球嗎？",
    "你幫我倒垃圾好嗎？",
    "我今天心情很不好",
    "你根本不在乎我！",
    "剛考完試，好開心！",
    "我們今晚吃什麼？",
    "AI怎麼回答我問題的？",
]

print("=" * 50)
print("💬 北爛男友對話測試")
print("=" * 50)

conversation_history = []  # 清空歷史

for msg in test_inputs:
    print(f"\n👩 女友：{msg}")
    reply = chat(msg)
    print(f"🦥 男友：{reply}")

print("\n" + "=" * 50)
print("\n🧠 記憶系統目前狀態：")
print(memory_to_text(memory))

'\ntest_inputs = [\n    "我叫小美，你在幹嘛？",\n    "你有沒有想我？等等要一起打球嗎？",\n    "你幫我倒垃圾好嗎？",\n    "我今天心情很不好",\n    "你根本不在乎我！",\n    "剛考完試，好開心！",\n    "我們今晚吃什麼？",\n    "AI怎麼回答我問題的？",\n]\n\nprint("=" * 50)\nprint("💬 北爛男友對話測試")\nprint("=" * 50)\n\nconversation_history = []  # 清空歷史\n\nfor msg in test_inputs:\n    print(f"\n👩 女友：{msg}")\n    reply = chat(msg)\n    print(f"🦥 男友：{reply}")\n\nprint("\n" + "=" * 50)\nprint("\n🧠 記憶系統目前狀態：")\nprint(memory_to_text(memory))\n'

In [7]:
import gradio as gr


def gradio_chat(message: str, history):
    """Gradio 對話函式"""
    reply = chat(message)
    return reply


def get_memory_display():
    """顯示目前記憶狀態"""
    return memory_to_text(memory)


def clear_all():
    """清空對話記錄與記憶"""
    global memory, conversation_history
    conversation_history = []
    memory = reset_memory()
    return [], "記憶已清空"


# ──────────────────────────────────────────
# Gradio 介面設計
# ──────────────────────────────────────────
with gr.Blocks(
    title="💤 北爛男友回覆系統",
    theme=gr.themes.Soft(),
    css="""
        .title { text-align: center; font-size: 2em; font-weight: bold; margin-bottom: 0.2em; }
        .subtitle { text-align: center; color: #888; margin-bottom: 1em; }
    """,
) as demo:

    gr.HTML("<div class='title'>💤 北爛男友回覆系統</div>")
    gr.HTML("<div class='subtitle'>他很懶，還有點靠北，但偶爾會甜一下 ❤️ — Powered by Groq</div>")

    with gr.Row():
        # 左側：對話區
        with gr.Column(scale=3):
            chatbot = gr.ChatInterface(
                fn=gradio_chat,
                chatbot=gr.Chatbot(
                    height=480,
                    placeholder="<div style='text-align:center;color:#aaa;margin-top:3em'>傳訊息給你的北爛男友吧 💬</div>",
                    avatar_images=(
                        None,
                        "https://api.dicebear.com/7.x/bottts/svg?seed=boyfriend&backgroundColor=b6e3f4",
                    ),
                ),
                textbox=gr.Textbox(
                    placeholder="說點什麼吧...",
                    container=False,
                ),
                examples=[
                    "你在幹嘛？",
                    "你有沒有想我？",
                    "我叫小美，我最愛吃拉麵",
                    "你幫我倒垃圾好嗎？",
                    "你根本不在乎我！",
                    "你愛我嗎？",
                    "我們今晚吃什麼？",
                    "我叫什麼名字你還記得嗎？",
                ],
                submit_btn="傳送 📱",
            )

        # 右側：記憶面板
        with gr.Column(scale=1):
            gr.Markdown("### 🧠 男友記憶庫")
            gr.Markdown("系統自動從對話中學習你的資訊")

            memory_display = gr.Textbox(
                label="目前已記住的事情",
                value=get_memory_display(),
                lines=14,
                interactive=False,
            )

            refresh_btn = gr.Button("🔄 更新記憶顯示", variant="secondary")
            clear_btn = gr.Button("🗑️ 清空全部記憶", variant="stop")

            refresh_btn.click(fn=get_memory_display, outputs=memory_display)
            clear_btn.click(
                fn=clear_all,
                outputs=[gr.State(), memory_display],
            )

    gr.Markdown(
        """---
**💡 使用提示：**
- 告訴他你的名字、生日、興趣，他會記住
- 之後隨時問「我叫什麼名字？」測試記憶
- 每次對話後點「更新記憶顯示」查看記憶狀態
"""
    )

demo.launch(share=True, show_error=True)
print("\n🌐 Gradio 介面已啟動！")

C:\Users\呂宗達\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\呂宗達\AppData\Local\Temp\ipykernel_11920\2682698265.py:26: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(
C:\Users\呂宗達\AppData\Local\Temp\ipykernel_11920\2682698265.py:41: UserWarning: You provided a custom `textbox` component, but also specified `submit_btn` parameter(s) on `gr.ChatInterface`. These ChatInterface parameters will be ignored. To customize these settings, pass them directly to your `gr.Textbox` or `gr.MultimodalTextbox` component instead. For example: textbox=gr.Textbox(..., submit_btn='s

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://c81f6d62ce5e87010f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🌐 Gradio 介面已啟動！
